In [1]:
# 09_model_blending.ipynb
# -------------------------------------------------------
# Blends multiple regression models for more stable predictions
# Uses weighted and simple averaging ensemble methods.

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")

# =====================================================
# Configuration
# =====================================================
project_root = Path("C:/JupyterProjects/Stock_ML_Project")
data_dir = project_root / "Data" / "Processed" / "enhanced"
results_dir = project_root / "Results"
results_dir.mkdir(parents=True, exist_ok=True)

tickers = {
    "RELIANCE": data_dir / "reliance_enhanced_model_ready.csv",
    "TCS": data_dir / "tcs_enhanced_model_ready.csv",
    "HDFCBANK": data_dir / "hdfcbank_enhanced_model_ready.csv"
}

# =====================================================
# Helper
# =====================================================
def evaluate(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

# =====================================================
# Blending Logic
# =====================================================
base_models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.001),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42)
}

results = []

for ticker, path in tickers.items():
    print(f"\n=== Processing {ticker} ===")
    if not path.exists():
        print(f"⚠️ Missing file: {path}")
        continue

    df = pd.read_csv(path)
    print(f"  Loaded: {df.shape}")

    target_col = "Target_Reg"
    if target_col not in df.columns:
        print(f"⚠️ Skipping {ticker} — no Target_Reg column found.")
        continue

    # Select numeric features only
    numeric_df = df.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)
    imputer = SimpleImputer(strategy="mean")
    numeric_df[numeric_df.columns] = imputer.fit_transform(numeric_df)

    X = numeric_df.drop(columns=[target_col], errors="ignore")
    y = numeric_df[target_col]

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.2)

    preds_df = pd.DataFrame(index=y_test.index)

    # --- Train base models ---
    for name, model in base_models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        preds_df[name] = preds

        metrics = evaluate(y_test, preds)
        metrics.update({"Ticker": ticker, "Model": name})
        results.append(metrics)
        print(f"  → {name}: RMSE={metrics['RMSE']:.3f}, R2={metrics['R2']:.3f}")

    # --- Blend Predictions ---
    preds_df["Blend_Simple"] = preds_df.mean(axis=1)

    # Weighted average favoring top models (Linear + Ridge + GB)
    preds_df["Blend_Weighted"] = (
        0.4 * preds_df["Linear"]
        + 0.25 * preds_df["Ridge"]
        + 0.15 * preds_df["GradientBoosting"]
        + 0.1 * preds_df["RandomForest"]
        + 0.1 * preds_df["DecisionTree"]
    )

    # Evaluate blends
    for blend in ["Blend_Simple", "Blend_Weighted"]:
        blend_metrics = evaluate(y_test, preds_df[blend])
        blend_metrics.update({"Ticker": ticker, "Model": blend})
        results.append(blend_metrics)
        print(f"  ✅ {blend}: RMSE={blend_metrics['RMSE']:.3f}, R2={blend_metrics['R2']:.3f}")

# =====================================================
# Save Results
# =====================================================
if results:
    results_df = pd.DataFrame(results)
    save_path = results_dir / "ensemble_blending_results.csv"
    results_df.to_csv(save_path, index=False)
    print(f"\n✅ Ensemble blending completed. Results saved to: {save_path}")
    display(results_df.sort_values(["Ticker", "R2"], ascending=[True, False]))
else:
    print("\n⚠️ No results generated.")



=== Processing RELIANCE ===
  Loaded: (1460, 25)
  → Linear: RMSE=17.585, R2=0.977
  → Ridge: RMSE=19.448, R2=0.972
  → Lasso: RMSE=17.960, R2=0.976
  → DecisionTree: RMSE=197.705, R2=-1.890
  → RandomForest: RMSE=180.245, R2=-1.402
  → GradientBoosting: RMSE=172.666, R2=-1.205
  ✅ Blend_Simple: RMSE=92.977, R2=0.361
  ✅ Blend_Weighted: RMSE=66.010, R2=0.678

=== Processing TCS ===
  Loaded: (1460, 25)
  → Linear: RMSE=52.021, R2=0.971
  → Ridge: RMSE=59.556, R2=0.962
  → Lasso: RMSE=55.021, R2=0.967
  → DecisionTree: RMSE=652.852, R2=-3.576
  → RandomForest: RMSE=539.876, R2=-2.129
  → GradientBoosting: RMSE=510.568, R2=-1.799
  ✅ Blend_Simple: RMSE=293.122, R2=0.078
  ✅ Blend_Weighted: RMSE=209.742, R2=0.528

=== Processing HDFCBANK ===
  Loaded: (1460, 25)
  → Linear: RMSE=9.567, R2=0.977
  → Ridge: RMSE=11.398, R2=0.968
  → Lasso: RMSE=10.072, R2=0.975
  → DecisionTree: RMSE=40.568, R2=0.591
  → RandomForest: RMSE=35.315, R2=0.690
  → GradientBoosting: RMSE=33.874, R2=0.715
  ✅ Bl

,RMSE,MAE,R2,Ticker,Model
16,9.566634,6.859892,0.977250,HDFCBANK,Linear
18,10.071847,7.172402,0.974784,HDFCBANK,Lasso
17,11.397678,8.551675,0.967708,HDFCBANK,Ridge
23,16.060325,11.479824,0.935884,HDFCBANK,Blend_Weighted
22,20.301077,14.179489,0.897554,HDFCBANK,Blend_Simple
21,33.873945,22.140068,0.714774,HDFCBANK,GradientBoosting
20,35.314659,23.711826,0.689995,HDFCBANK,RandomForest
19,40.568369,29.931323,0.590896,HDFCBANK,DecisionTree
0,17.585252,12.737170,0.977133,RELIANCE,Linear
2,17.960401,12.867725,0.976146,RELIANCE,Lasso
